In [ ]:
"""
==============================================================================
RETAIL CREDIT PROJECT (BRAZIL) - STAGE 1 CONSOLIDATED MASTER
Senior Quant Review: Denominator Correction (Total Exposure), 
Official Geographic Meshes (IBGE/geobr) and Shielded Risk Matrices.
Complete Integration: Generation of all charts and 2x2 panels.
Update: Macroeconomic Regimes Disaggregation (Table 3) 
Author: Itaiguara de Oliveira Bezerra
==============================================================================
"""

import os
import warnings
from typing import Optional

import geopandas as gpd
import geobr  # IBGE Official Library
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
import pandas as pd

warnings.filterwarnings("ignore")

plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#4a6175"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["grid.color"] = "#dcdcdc"
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["grid.alpha"] = 0.5
pd.set_option("display.float_format", lambda x: "%.4f" % x)

def remover_molduras(ax: Optional[plt.Axes] = None) -> None:
    if ax is None:
        ax = plt.gca()
    for spine in ["top", "right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

# ------------------------------------------------------------------------------
# 1. METHODOLOGICAL PARAMETERS AND OFFICIAL TAXONOMY
# ------------------------------------------------------------------------------
MODALIDADES_EXPURGO = [
    "Financiamentos à exportação",
    "Financiamentos à importação",
    "Financiamentos de infraestrutura e desenvolvimento",
    "Financiamentos de títulos e valores mobiliários",
    "Financiamentos com interveniência",
]

MAPA_REGIOES_DICT = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte", "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste", "PB": "Nordeste", "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul",
}

NOMES_ESTADOS_BR = {
    "AC": "Acre", "AL": "Alagoas", "AP": "Amapá", "AM": "Amazonas", "BA": "Bahia", "CE": "Ceará", "DF": "Distrito Federal",
    "ES": "Espírito Santo", "GO": "Goiás", "MA": "Maranhão", "MT": "Mato Grosso", "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais", "PA": "Pará", "PB": "Paraíba", "PR": "Paraná", "PE": "Pernambuco", "PI": "Piauí",
    "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte", "RS": "Rio Grande do Sul", "RO": "Rondônia",
    "RR": "Roraima", "SC": "Santa Catarina", "SP": "São Paulo", "SE": "Sergipe", "TO": "Tocantins",
}

ORDEM_REGIOES = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]

# Translation Map for the final table formatting
REGION_TRANSLATION = {
    "Norte": "North Region",
    "Nordeste": "Northeast Region",
    "Centro-Oeste": "Midwest Region",
    "Sudeste": "Southeast Region",
    "Sul": "South Region"
}

def mapear_macrobloco_pf_vetorizado(df: pd.DataFrame) -> np.ndarray:
    sub = df["submodalidade"].astype(str).str.lower()
    mod = df["modalidade"].astype(str).str.lower()

    cond_rotativo = sub.str.contains("crédito rotativo vinculado a cartão|rotativo|cheque especial|sem consignação|adiantamentos a depositantes|conta garantida|fatura parcelada|saque financiado|antecipação de fatura", regex=True, na=False)
    cond_consignado = sub.str.contains("com consignação|consignado", regex=True, na=False)
    cond_garantia_real = (sub.str.contains("habitacional|sfh|veículos automotores|veículos autom.|home equity|veiculo|veículo|arrendamento financeiro de veículos", regex=True, na=False) | mod.str.contains("imobiliários", regex=True, na=False))

    choices = ["Revolving / Unsecured", "Payroll / Secured", "Real Collateral (Mortgage/Auto)"]
    conditions = [cond_rotativo, cond_consignado, cond_garantia_real]
    return np.select(conditions, choices, default="Others / Renegotiated")

def estilizar_tabela_corporativa(tabela):
    for (row_idx, col_idx), cell in tabela.get_celld().items():
        cell.set_edgecolor("#dcdcdc")
        cell.set_linewidth(0.5)
        if row_idx == 0:
            cell.set_facecolor("#2c3e50")
            cell.set_text_props(color="#ffffff", weight="bold")
            cell.set_height(0.038)
        else:
            cell.set_facecolor("#f8f9fa" if row_idx % 2 == 0 else "#ffffff")
            if col_idx == 0: cell.set_text_props(ha="left")

# ------------------------------------------------------------------------------
# 2. PARQUET DATABASE LOADING AND CLEANSING
# ------------------------------------------------------------------------------
def main():
    print("=" * 95)
    print("STAGE 1 (BRAZIL): MULTIDIMENSIONAL PROCESSING AND TERRITORIAL APPENDIX")
    print("=" * 95)

    arquivo_parquet = "painel_scr_pf_bruto.parquet"
    if not os.path.exists(arquivo_parquet):
        raise FileNotFoundError(f"File '{arquivo_parquet}' not found. Complete Stage 0 first.")

    cols_load = [
        "data_base", "uf", "cnae_ocupacao", "porte", "modalidade", "submodalidade", 
        "carteira_ativa", "vencido_acima_de_90_dias", "ativo_problematico"
    ]

    print(f">>> Reading strategic columns from '{arquivo_parquet}'...")
    df = pd.read_parquet(arquivo_parquet, columns=cols_load)
    df["data_base"] = pd.to_datetime(df["data_base"], errors="coerce")

    df["carteira_total"] = df["carteira_ativa"].fillna(0) + df["vencido_acima_de_90_dias"].fillna(0)

    n_antes = len(df)
    df = df[~df["modalidade"].isin(MODALIDADES_EXPURGO)].copy()
    print(f"[i] Corporate modalities purged: {n_antes - len(df):,} records removed.")

    df["macrobloco"] = mapear_macrobloco_pf_vetorizado(df)
    df["regiao"] = df["uf"].map(MAPA_REGIOES_DICT)

    # ------------------------------------------------------------------------------
    # 3. VECTORIZED STATISTICAL DESCRIPTIVE ENGINE
    # ------------------------------------------------------------------------------
    def calcular_tabela_descritiva(df_base, col_dimensao, nome_dimensao):
        ts = (
            df_base.groupby(["data_base", col_dimensao])
            .agg({
                "carteira_total": "sum",
                "vencido_acima_de_90_dias": "sum",
                "ativo_problematico": "sum",
            })
            .reset_index()
        )

        ts["inad_taxa"] = (ts["vencido_acima_de_90_dias"] / ts["carteira_total"]) * 100
        ts["prob_taxa"] = (ts["ativo_problematico"] / ts["carteira_total"]) * 100

        n_datas = len(ts["data_base"].unique())
        saldo_total_medio_sistema = (df_base["carteira_total"].sum() / (n_datas * 1e9) if n_datas > 0 else 1.0)

        linhas = []
        for grupo, grp in ts.groupby(col_dimensao):
            if len(grp) >= 6:
                s_inad = grp["inad_taxa"].dropna()
                s_prob = grp["prob_taxa"].dropna()
                s_vol = (grp["carteira_total"] / 1e9).dropna()

                linhas.append({
                    nome_dimensao: str(grupo),
                    "Avg Balance (BRL Bn)": s_vol.mean(),
                    "Avg Share (%)": (s_vol.mean() / saldo_total_medio_sistema) * 100,
                    "Avg Def (%)": s_inad.mean(),
                    "Def Std (%)": s_inad.std(),
                    "Def Min (%)": s_inad.min(),
                    "Def Median (%)": s_inad.median(),
                    "Def Max (%)": s_inad.max(),
                    "Def Skew": s_inad.skew() if len(s_inad) >= 3 else 0.0,
                    "Def Kurt": s_inad.kurtosis() if len(s_inad) >= 4 else 0.0,
                    "Avg Prob Assets (%)": s_prob.mean(),
                })
        return pd.DataFrame(linhas).sort_values("Avg Balance (BRL Bn)", ascending=False)

    print("\n>>> Processing Multidimensional Descriptive Tables...")
    tabela_macroblocos = calcular_tabela_descritiva(df, "macrobloco", "Credit Modality")
    
    # Translate Occupations for English Table
    df["cnae_ocupacao_en"] = df["cnae_ocupacao"].replace({
        "Aposentado/pensionista": "Retiree / Pensioner",
        "Servidor ou empregado público": "Public Servant",
        "Empregado de empresa privada": "Private Sector Employee",
        "Autônomo": "Self-Employed",
        "MEI": "Microentrepreneur (MEI)",
        "Outros": "Others",
        "Empregado de ent. sem fins lucrativos": "Non-Profit Employee"
    })
    tabela_ocupacao = calcular_tabela_descritiva(df, "cnae_ocupacao_en", "Borrower Occupational Profile")
    
    # Translate Income Brackets
    df["porte_en"] = df["porte"].replace({
        "Até 1 salário mínimo": "Up to 1 Minimum Brazilian Wage",
        "Mais de 1 a 2 salários mínimos": "1 to 2 Minimum Brazilian Wages",
        "Mais de 2 a 3 salários mínimos": "2 to 3 Minimum Brazilian Wages",
        "Mais de 3 a 5 salários mínimos": "3 to 5 Minimum Brazilian Wages",
        "Mais de 5 a 10 salários mínimos": "5 to 10 Minimum Brazilian Wages",
        "Mais de 10 a 20 salários mínimos": "10 to 20 Minimum Brazilian Wages",
        "Acima de 20 salários mínimos": "Above 20 Minimum Brazilian Wages",
        "Indisponível": "Unavailable",
        "Sem rendimento": "No Income"
    })
    tabela_porte = calcular_tabela_descritiva(df, "porte_en", "Borrower Income Bracket")
    
    df["regiao_en"] = df["regiao"].map(REGION_TRANSLATION)
    tabela_regiao = calcular_tabela_descritiva(df, "regiao_en", "Geographic Region")

    ts_macro = (
        df.groupby("data_base")
        .agg({
            "carteira_total": "sum",
            "vencido_acima_de_90_dias": "sum",
            "ativo_problematico": "sum",
        })
        .reset_index().sort_values("data_base")
    )
    ts_macro["taxa_inadimplencia_pct"] = (ts_macro["vencido_acima_de_90_dias"] / ts_macro["carteira_total"]) * 100
    ts_macro["taxa_problematico_pct"] = (ts_macro["ativo_problematico"] / ts_macro["carteira_total"]) * 100
    ts_macro["saldo_ativo_tri"] = ts_macro["carteira_total"] / 1e12

    # ------------------------------------------------------------------------------
    # 4. CROSS-TABS
    # ------------------------------------------------------------------------------
    print(">>> Generating Cross-Matrices: Intracell Rates and Relative Distributions...")

    cruz_ocup_total = (df.pivot_table(index="macrobloco", columns="cnae_ocupacao_en", values="carteira_total", aggfunc="sum") / 1e9).fillna(0.0)
    cruz_ocup_venc = (df.pivot_table(index="macrobloco", columns="cnae_ocupacao_en", values="vencido_acima_de_90_dias", aggfunc="sum") / 1e9).fillna(0.0)
    cruz_renda_total = (df.pivot_table(index="macrobloco", columns="porte_en", values="carteira_total", aggfunc="sum") / 1e9).fillna(0.0)
    cruz_renda_venc = (df.pivot_table(index="macrobloco", columns="porte_en", values="vencido_acima_de_90_dias", aggfunc="sum") / 1e9).fillna(0.0)

    cruz_ocup_inad = ((cruz_ocup_venc / cruz_ocup_total) * 100).fillna(0.0)
    cruz_renda_inad = ((cruz_renda_venc / cruz_renda_total) * 100).fillna(0.0)
    dist_inad_ocupacao = (cruz_ocup_venc.div(cruz_ocup_venc.sum(axis=0), axis=1) * 100).fillna(0.0)
    dist_inad_renda = (cruz_renda_venc.div(cruz_renda_venc.sum(axis=0), axis=1) * 100).fillna(0.0)

    # ------------------------------------------------------------------------------
    # 6. EXCEL EXPORT (FORMATTED)
    # ------------------------------------------------------------------------------
    nome_excel_exec = "executive_report_stage1_retail_en.xlsx"
    with pd.ExcelWriter(nome_excel_exec, engine="openpyxl") as writer:
        tabela_macroblocos.to_excel(writer, sheet_name="1_Macroblocks", index=False)
        tabela_ocupacao.to_excel(writer, sheet_name="2_Occupational_Profile", index=False)
        tabela_porte.to_excel(writer, sheet_name="3_Income_Brackets", index=False)
        tabela_regiao.to_excel(writer, sheet_name="4_Regions", index=False)
        cruz_ocup_inad.reset_index().to_excel(writer, sheet_name="Cross_Def_Occupation", index=False)
        cruz_renda_inad.reset_index().to_excel(writer, sheet_name="Cross_Def_Income", index=False)
        dist_inad_ocupacao.reset_index().to_excel(writer, sheet_name="Dist_Def_Occupation", index=False)
        dist_inad_renda.reset_index().to_excel(writer, sheet_name="Dist_Def_Income", index=False)
        ts_macro.to_excel(writer, sheet_name="Macro_Series_Retail", index=False)

    wb_exec = openpyxl.load_workbook(nome_excel_exec)
    header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
    header_font = Font(name="Arial", size=10, bold=True, color="FFFFFF")
    thin_border = Border(left=Side(style="thin", color="D9D9D9"), right=Side(style="thin", color="D9D9D9"), top=Side(style="thin", color="D9D9D9"), bottom=Side(style="thin", color="D9D9D9"))

    abas_com_percentual = ["Cross_Def_Occupation", "Cross_Def_Income", "Dist_Def_Occupation", "Dist_Def_Income"]

    for ws in wb_exec.worksheets:
        for col_idx in range(1, ws.max_column + 1):
            c_h = ws.cell(row=1, column=col_idx)
            c_h.fill = header_fill
            c_h.font = header_font
            c_h.alignment = Alignment(horizontal="center", vertical="center")
        for row_idx in range(2, ws.max_row + 1):
            for col_idx in range(1, ws.max_column + 1):
                cell = ws.cell(row=row_idx, column=col_idx)
                cell.border = thin_border
                if ws.title in abas_com_percentual and col_idx > 1:
                    cell.number_format = '0.00"%"'
                    cell.alignment = Alignment(horizontal="right", vertical="center")
    wb_exec.save(nome_excel_exec)
    print(f"[OK] Complete Executive Report saved: '{nome_excel_exec}'")

    # ------------------------------------------------------------------------------
    # 7. PANEL 1: REGIONAL MAP (GEOBR)
    # ------------------------------------------------------------------------------
    print("\n--- GENERATING PANEL 1: REGIONAL MAP AND MACROBLOCKS (EDITORIAL) ---")
    try:
        resumo_reg_total = df.groupby("regiao").agg(carteira_total=("carteira_total", "sum"), vencido_total=("vencido_acima_de_90_dias", "sum")).reset_index()
        resumo_reg_total["inadimplencia_total_pct"] = (resumo_reg_total["vencido_total"] / resumo_reg_total["carteira_total"]) * 100

        resumo_reg_macro = df.groupby(["regiao", "macrobloco"]).agg(carteira_bloco=("carteira_total", "sum"), vencido_bloco=("vencido_acima_de_90_dias", "sum")).reset_index()
        resumo_reg_macro["inadimplencia_bloco_pct"] = (resumo_reg_macro["vencido_bloco"] / resumo_reg_macro["carteira_bloco"]) * 100

        gdf_estados = geobr.read_state(year=2020)
        gdf_estados = gdf_estados.rename(columns={"abbrev_state": "UF"})

        gdf_estados["regiao"] = gdf_estados["UF"].map(MAPA_REGIOES_DICT)
        gdf_regioes = gdf_estados.dissolve(by="regiao", as_index=False)
        gdf_final = gdf_regioes.merge(resumo_reg_total, on="regiao", how="left")

        fig_mapa = plt.figure(figsize=(19, 11), dpi=300)
        gs_geo = gridspec.GridSpec(5, 2, figure=fig_mapa, width_ratios=[1.15, 0.85], wspace=0.25, hspace=0.45)
        ax_mapa = fig_mapa.add_subplot(gs_geo[:, 0])

        cores_macroblocos = {"Revolving / Unsecured": "#d62728", "Payroll / Secured": "#2ca02c", "Real Collateral (Mortgage/Auto)": "#1f77b4", "Others / Renegotiated": "#ff7f0e"}

        gdf_final.plot(column="inadimplencia_total_pct", cmap="YlOrRd", linewidth=1.2, edgecolor="#4a4a4a", legend=True, legend_kwds={"label": "Aggregate Retail Default Rate (> 90 days) %", "orientation": "horizontal", "shrink": 0.65, "pad": 0.02}, ax=ax_mapa)
        gdf_estados.plot(ax=ax_mapa, facecolor="none", edgecolor="#ffffff", linewidth=0.6, alpha=0.6)

        offset_coords = {"Norte": (-56.0, -3.5), "Nordeste": (-40.5, -8.5), "Centro-Oeste": (-54.0, -15.5), "Sudeste": (-43.5, -20.5), "Sul": (-52.0, -28.0)}

        for _, r_row in gdf_final.iterrows():
            reg_nome = str(r_row["regiao"])
            reg_nome_en = REGION_TRANSLATION.get(reg_nome, reg_nome).replace(" Region", "")
            taxa_val = r_row["inadimplencia_total_pct"]
            x_c, y_c = offset_coords.get(reg_nome, (r_row.geometry.centroid.x, r_row.geometry.centroid.y))
            ax_mapa.text(x_c, y_c, f"{reg_nome_en}\n{taxa_val:.2f}%", ha="center", va="center", fontsize=10.5, fontweight="bold", color="#16222f", bbox=dict(boxstyle="round,pad=0.45", facecolor="#ffffff", edgecolor="#4a6175", alpha=0.92, lw=0.8))

        ax_mapa.set_axis_off()
        ax_mapa.set_title("")

        for i, reg_nome in enumerate(ORDEM_REGIOES):
            ax_sub = fig_mapa.add_subplot(gs_geo[i, 1])
            sub_bloco = resumo_reg_macro[resumo_reg_macro["regiao"] == reg_nome].sort_values("inadimplencia_bloco_pct", ascending=True)
            cores_barras = [cores_macroblocos.get(b, "#1f77b4") for b in sub_bloco["macrobloco"]]
            bars_reg = ax_sub.barh(sub_bloco["macrobloco"], sub_bloco["inadimplencia_bloco_pct"], color=cores_barras, height=0.6)
            max_v = sub_bloco["inadimplencia_bloco_pct"].max()
            for b_reg in bars_reg:
                w_reg = b_reg.get_width()
                ax_sub.text(w_reg + 0.15, b_reg.get_y() + b_reg.get_height() / 2, f"{w_reg:.2f}%", va="center", ha="left", fontsize=8.5, fontweight="bold", color="#222222")
            ax_sub.set_xlim(0, max_v * 1.25 if max_v > 0 else 1.0)
            reg_title = REGION_TRANSLATION.get(reg_nome, reg_nome)
            ax_sub.set_title(f"{reg_title}", fontsize=10, fontweight="bold", pad=4, color="#16222f")
            remover_molduras(ax_sub)

        # NOME EXATO FORNECIDO
        nome_painel_reg = "panel_stage1_retail_regional_macroblocks.png"
        plt.savefig(nome_painel_reg, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"[OK] Regional Map Panel saved: '{nome_painel_reg}'")
    except Exception as e:
        print(f"[!] Geopandas plotting failed: {e}")

    # ------------------------------------------------------------------------------
    # 8. PANEL 2: TERRITORIAL APPENDIX (STATE MAP + TABLE)
    # ------------------------------------------------------------------------------
    print("\n--- GENERATING PANEL 2: TERRITORIAL APPENDIX (EDITORIAL) ---")
    try:
        df_uf_macro = df.groupby(["regiao", "uf", "macrobloco"]).agg(carteira=("carteira_total", "sum"), vencido=("vencido_acima_de_90_dias", "sum")).reset_index()
        df_uf_macro["taxa_bloco_pct"] = (df_uf_macro["vencido"] / df_uf_macro["carteira"]) * 100

        idx_max = df_uf_macro.groupby(["regiao", "uf"])["taxa_bloco_pct"].idxmax().dropna()
        tabela_critica = df_uf_macro.loc[idx_max, ["regiao", "uf", "macrobloco", "taxa_bloco_pct"]].copy()

        df_uf_geral = df.groupby("uf").agg(carteira_uf=("carteira_total", "sum"), vencido_uf=("vencido_acima_de_90_dias", "sum")).reset_index()
        df_uf_geral["taxa_uf_pct"] = (df_uf_geral["vencido_uf"] / df_uf_geral["carteira_uf"]) * 100

        df_apendice = tabela_critica.merge(df_uf_geral[["uf", "taxa_uf_pct"]], on="uf")
        df_apendice.rename(columns={"regiao": "Macro-region", "uf": "UF", "macrobloco": "Main Driver", "taxa_bloco_pct": "Def. (%)", "taxa_uf_pct": "Avg. (%)"}, inplace=True)
        df_apendice["State"] = df_apendice["UF"].map(NOMES_ESTADOS_BR)
        df_apendice = df_apendice.sort_values("Avg. (%)", ascending=False).reset_index(drop=True)

        fig_ap = plt.figure(figsize=(18, 12), dpi=300)
        gs_ap = fig_ap.add_gridspec(1, 2, width_ratios=[1.1, 1.1], wspace=0.15, left=0.04, right=0.96)

        ax_mapa_uf = fig_ap.add_subplot(gs_ap[0, 0])
        gdf_mapa_uf = gdf_estados.merge(df_apendice[["UF", "Avg. (%)"]], on="UF", how="inner")
        gdf_mapa_uf.plot(column="Avg. (%)", cmap="YlOrRd", linewidth=0.8, ax=ax_mapa_uf, edgecolor="#4a6175", legend=True, legend_kwds={"label": "Overall Effective Default Average by State (%)", "orientation": "horizontal", "shrink": 0.70, "pad": 0.04})

        for _, row in gdf_mapa_uf.iterrows():
            pt = row.geometry.representative_point()
            ax_mapa_uf.text(pt.x, pt.y, row["UF"], fontsize=7.5, ha="center", va="center", fontweight="bold", color="#16222f")
        ax_mapa_uf.set_title("")
        ax_mapa_uf.axis("off")

        ax_tab = fig_ap.add_subplot(gs_ap[0, 1])
        ax_tab.axis("off")

        dados_tabela_unica = []
        for _, r in df_apendice.iterrows():
            dados_tabela_unica.append([r["State"], r["UF"], str(r["Main Driver"]), f"{r['Def. (%)']:.2f}%", f"{r['Avg. (%)']:.2f}%"])

        t_unica = ax_tab.table(cellText=dados_tabela_unica, colLabels=["State", "Code", "Main Driver", "Def. (%)", "Avg. (%)"], cellLoc="center", loc="center", colWidths=[0.32, 0.12, 0.26, 0.15, 0.15])
        t_unica.auto_set_font_size(False)
        t_unica.set_fontsize(8.5)
        t_unica.scale(1.0, 1.28)

        estilizar_tabela_corporativa(t_unica)
        
        # NOME EXATO FORNECIDO
        nome_painel_apendice = "panel_appendix_retail_states_complete.png"
        plt.savefig(nome_painel_apendice, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"[OK] State Appendix Panel saved: '{nome_painel_apendice}'")

    except Exception as e:
        print(f"[!] Territorial Appendix plotting failed: {e}")


    # ------------------------------------------------------------------------------
    # 10. 2x2 PANELS: BALANCE EVOLUTION AND MULTIDIMENSIONAL EDA
    # ------------------------------------------------------------------------------
    print("\n--- GENERATING 2x2 PANELS: HISTORICAL BALANCES AND EDA (EDITORIAL) ---")

    ts_saldos = df.groupby(["data_base", "macrobloco"])["carteira_total"].sum().reset_index()
    ts_saldos["saldo_tri"] = ts_saldos["carteira_total"] / 1e12

    macroblocos_ordem = [("Revolving / Unsecured", "#d62728"), ("Payroll / Secured", "#1f77b4"), ("Real Collateral (Mortgage/Auto)", "#2ca02c"), ("Others / Renegotiated", "#ff7f0e")]

    fig_saldo, axes_saldo = plt.subplots(2, 2, figsize=(14, 9), dpi=300)
    for idx, (mb, cor) in enumerate(macroblocos_ordem):
        ax = axes_saldo.flatten()[idx]
        df_mb = ts_saldos[ts_saldos["macrobloco"] == mb].sort_values("data_base")
        ax.plot(df_mb["data_base"], df_mb["saldo_tri"], color=cor, lw=2.2)
        ax.fill_between(df_mb["data_base"], df_mb["saldo_tri"], color=cor, alpha=0.1)
        ax.axvline(pd.to_datetime("2020-01-01"), color="gray", linestyle=":", lw=1.2, alpha=0.8)
        ax.set_title(mb, fontsize=11, fontweight="bold", pad=8, color="#16222f")
        ax.set_xlabel("Year", fontsize=9.5, fontweight="bold")
        ax.set_ylabel("Total Exposure Balance (BRL Trillions)", fontsize=9.5, fontweight="bold")
        ax.grid(True)
        ymax = df_mb["saldo_tri"].max()
        ax.set_ylim(0, ymax * 1.15 if ymax > 0 else 1)
        remover_molduras(ax)

    plt.tight_layout()
    
    # NOME EXATO FORNECIDO
    nome_painel_saldos = "panel_stage1_retail_balance_evolution_4macroblocks.png"
    plt.savefig(nome_painel_saldos, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[OK] Balance Panel saved: '{nome_painel_saldos}'")

    fig_eda, axes_eda = plt.subplots(2, 2, figsize=(15, 11), dpi=300)

    ts_macro_b = df.groupby(["data_base", "macrobloco"]).agg({"carteira_total": "sum", "vencido_acima_de_90_dias": "sum"}).reset_index()
    ts_macro_b["inad_pct"] = (ts_macro_b["vencido_acima_de_90_dias"] / ts_macro_b["carteira_total"]) * 100
    for mb, grp in ts_macro_b.groupby("macrobloco"):
        axes_eda[0, 0].plot(grp["data_base"], grp["inad_pct"], lw=2, label=mb)
    axes_eda[0, 0].set_title("Default by Credit Line", fontsize=10.5, fontweight="bold", pad=8, color="#16222f")
    axes_eda[0, 0].set_xlabel("Year", fontsize=9.5, fontweight="bold"); axes_eda[0, 0].set_ylabel("Default Rate (%)", fontsize=10, fontweight="bold")
    axes_eda[0, 0].grid(True); axes_eda[0, 0].legend(frameon=False, facecolor="#f8f9fa", fontsize=8.5); remover_molduras(axes_eda[0, 0])

    ts_ocupa = df.groupby(["data_base", "cnae_ocupacao_en"]).agg({"carteira_total": "sum", "vencido_acima_de_90_dias": "sum"}).reset_index()
    ts_ocupa["inad_pct"] = (ts_ocupa["vencido_acima_de_90_dias"] / ts_ocupa["carteira_total"]) * 100
    for oc in ["Retiree / Pensioner", "Public Servant", "Private Sector Employee", "Self-Employed", "Microentrepreneur (MEI)"]:
        grp = ts_ocupa[ts_ocupa["cnae_ocupacao_en"] == oc]
        if not grp.empty: axes_eda[0, 1].plot(grp["data_base"], grp["inad_pct"], lw=1.8, label=oc)
    axes_eda[0, 1].set_title("Default by Occupation Profile", fontsize=10.5, fontweight="bold", pad=8, color="#16222f")
    axes_eda[0, 1].set_xlabel("Year", fontsize=9.5, fontweight="bold"); axes_eda[0, 1].set_ylabel("Default Rate (%)", fontsize=10, fontweight="bold")
    axes_eda[0, 1].grid(True); axes_eda[0, 1].legend(frameon=False, facecolor="#f8f9fa", fontsize=8.5); remover_molduras(axes_eda[0, 1])

    axes_eda[1, 0].scatter(ts_macro["taxa_inadimplencia_pct"], ts_macro["taxa_problematico_pct"], color="#2b5c8f", s=45, alpha=0.7)
    axes_eda[1, 0].set_title("Dispersion: Default (>90d) vs. Total Problem Assets", fontsize=10.5, fontweight="bold", pad=8, color="#16222f")
    axes_eda[1, 0].set_xlabel("Default > 90 days (%)", fontsize=10, fontweight="bold"); axes_eda[1, 0].set_ylabel("Total Problem Assets (%)", fontsize=10, fontweight="bold")
    axes_eda[1, 0].grid(True); remover_molduras(axes_eda[1, 0])

    ts_porte = df.groupby(["data_base", "porte_en"]).agg({"carteira_total": "sum", "vencido_acima_de_90_dias": "sum"}).reset_index()
    ts_porte["inad_pct"] = (ts_porte["vencido_acima_de_90_dias"] / ts_porte["carteira_total"]) * 100
    for pt in ["Up to 1 Minimum Brazilian Wage", "1 to 2 Minimum Brazilian Wages", "2 to 3 Minimum Brazilian Wages", "5 to 10 Minimum Brazilian Wages", "Above 20 Minimum Brazilian Wages"]:
        grp = ts_porte[ts_porte["porte_en"] == pt]
        if not grp.empty: axes_eda[1, 1].plot(grp["data_base"], grp["inad_pct"], lw=1.8, label=pt)
    axes_eda[1, 1].set_title("Default by Borrower Income Bracket", fontsize=10.5, fontweight="bold", pad=8, color="#16222f")
    axes_eda[1, 1].set_xlabel("Year", fontsize=9.5, fontweight="bold"); axes_eda[1, 1].set_ylabel("Default Rate (%)", fontsize=10, fontweight="bold")
    axes_eda[1, 1].grid(True); axes_eda[1, 1].legend(frameon=False, facecolor="#f8f9fa", fontsize=8.5); remover_molduras(axes_eda[1, 1])

    plt.tight_layout()
    
    # NOME EXATO FORNECIDO COM "_2"
    nome_painel_eda = "panel_eda_stage1_retail_multidimensional_2.png"
    plt.savefig(nome_painel_eda, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[OK] EDA Multidimensional Panel saved: '{nome_painel_eda}'")

    # ------------------------------------------------------------------------------
    # 11. MACROECONOMIC REGIMES IMPACT (TABLE 3)
    # ------------------------------------------------------------------------------
    print("\n--- MACROECONOMIC REGIMES IMPACT ON BALANCES (Table 3) ---")
    ts_saldos_bi = ts_saldos.copy()
    ts_saldos_bi["saldo_bi"] = ts_saldos_bi["carteira_total"] / 1e9

    pre_choque = ts_saldos_bi[ts_saldos_bi["data_base"] <= '2020-02-29'].groupby('macrobloco')['saldo_bi'].mean()
    estimulo = ts_saldos_bi[(ts_saldos_bi["data_base"] >= '2020-03-01') & (ts_saldos_bi["data_base"] <= '2021-12-31')].groupby('macrobloco')['saldo_bi'].mean()
    aperto = ts_saldos_bi[(ts_saldos_bi["data_base"] >= '2022-01-01') & (ts_saldos_bi["data_base"] <= '2023-12-31')].groupby('macrobloco')['saldo_bi'].mean()
    normalizacao = ts_saldos_bi[ts_saldos_bi["data_base"] >= '2024-01-01'].groupby('macrobloco')['saldo_bi'].mean()

    var_estimulo = ((estimulo - pre_choque) / pre_choque) * 100
    var_aperto = ((aperto - estimulo) / estimulo) * 100
    var_normalizacao = ((normalizacao - aperto) / aperto) * 100

    resultado = pd.DataFrame({
        '1. Pre-Shock (BRL Bn)': pre_choque, 
        '2. Stimulus (BRL Bn)': estimulo, 
        'Var. Stimulus (%)': var_estimulo,
        '3. Tightening (BRL Bn)': aperto,
        'Var. Tightening (%)': var_aperto,
        '4. Normalization (BRL Bn)': normalizacao,
        'Var. Normalization (%)': var_normalizacao
    }).round(2)

    print(resultado)

    print("\n" + "=" * 95)
    print("[OK] Execution completed and files successfully saved.")
    print("=" * 95)

if __name__ == '__main__':
    main()